# 07. The Transformer Architecture ("Attention Is All You Need")

Published by Vaswani et al. in 2017, the Transformer architecture entirely replaced recurrent and convolutional layers with attention mechanisms, achieving unprecedented parallelization and scaling efficiency.

### Architectural Anatomy of a Transformer Block:
1. Input Embeddings + Positional Encodings
2. Multi-Head Self-Attention
3. Residual Add & Layer Normalization: $\text{LayerNorm}(\mathbf{x} + \text{Sublayer}(\mathbf{x}))$
4. Position-Wise Feed-Forward Network: $\text{FFN}(\mathbf{x}) = \max(0, \mathbf{x}\mathbf{W}_1 + \mathbf{b}_1)\mathbf{W}_2 + \mathbf{b}_2$
5. Residual Add & Layer Normalization

In [ ]:
import torch
import torch.nn as nn

class TransformerBlock(nn.Module):
    def __init__(self, d_model=64, n_heads=4, d_ff=128, dropout=0.1):
        super().__init__()
        self.attn = nn.MultiheadAttention(d_model, n_heads, batch_first=True)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.GELU(),
            nn.Linear(d_ff, d_model),
            nn.Dropout(dropout)
        )

    def forward(self, x):
        # Sublayer 1: Self-Attention + Residual
        attn_out, _ = self.attn(x, x, x)
        x = self.norm1(x + attn_out)
        # Sublayer 2: Feed-Forward + Residual
        x = self.norm2(x + self.ffn(x))
        return x

block = TransformerBlock()
x = torch.randn(2, 8, 64)
print("Transformer Block Output Shape:", block(x).shape)

## 🎯 Exercises & Practice
1. Explain the difference between Layer Normalization (used in Transformers) and Batch Normalization (used in CNNs).

In [ ]:
# Exercise Solution
# Answer: BatchNorm computes mean/variance across the batch dimension (fails with variable batch sizes/sequences).
# LayerNorm normalizes across the feature dimension for each token independently, making it invariant to batch size!